# What drives the price of a car?

![](images/kurt.jpeg)

**OVERVIEW**

In this application, you will explore a dataset from Kaggle. The original dataset contained information on 3 million used cars. The provided dataset contains information on 426K cars to ensure speed of processing.  Your goal is to understand what factors make a car more or less expensive.  As a result of your analysis, you should provide clear recommendations to your client -- a used car dealership -- as to what consumers value in a used car.

### CRISP-DM Framework

<center>
    <img src = images/crisp.png width = 50%/>
</center>


To frame the task, throughout our practical applications, we will refer back to a standard process in industry for data projects called CRISP-DM.  This process provides a framework for working through a data problem.  Your first step in this application will be to read through a brief overview of CRISP-DM [here](https://mo-pcco.s3.us-east-1.amazonaws.com/BH-PCMLAI/module_11/readings_starter.zip).  After reading the overview, answer the questions below.

### Business Understanding

From a business perspective, we are tasked with identifying key drivers for used car prices.  In the CRISP-DM overview, we are asked to convert this business framing to a data problem definition.  Using a few sentences, reframe the task as a data task with the appropriate technical vocabulary. 

### Data Understanding

After considering the business understanding, we want to get familiar with our data.  Write down some steps that you would take to get to know the dataset and identify any quality issues within.  Take time to get to know the dataset and explore what information it contains and how this could be used to inform your business understanding.

In [ ]:
From a data mining perspective, this task translates into a multivariate regression problem where the objective 
is to model the continuous target variable, price, as a function of independent feature vectors such as odometer,
 year, and manufacturer.

In [ ]:
The goal is to employ exploratory data analysis (EDA) to clean the dataset and select relevant attributes, 
followed by feature importance analysis (using techniques like correlation matrices or regression coefficients) 
to statistically quantify which features explain the most variance in the vehicle's price

### Data Preparation

After our initial exploration and fine-tuning of the business understanding, it is time to construct our final dataset prior to modeling.  Here, we want to make sure to handle any integrity issues and cleaning, the engineering of new features, any transformations that we believe should happen (scaling, logarithms, normalization, etc.), and general preparation for modeling with `sklearn`. 

In [1]:
import pandas as pd
import numpy as np

# 1. Load the Data
df = pd.read_csv('vehicles.csv')

# 2. Integrity Issues & Cleaning
# Drop columns that are irrelevant for the model or have too many missing values
columns_to_drop = ['id', 'VIN', 'region', 'size'] 
df_cleaned = df.drop(columns=columns_to_drop, errors='ignore')

# Drop rows where critical information is missing
df_cleaned = df_cleaned.dropna(subset=['year', 'odometer', 'manufacturer', 'model'])

# Handle Price Outliers (Removing unrealistic values like $0 or >$100k)
df_cleaned = df_cleaned[(df_cleaned['price'] > 500) & (df_cleaned['price'] < 100000)]

# Handle Odometer Outliers (Removing unrealistic mileage)
df_cleaned = df_cleaned[(df_cleaned['odometer'] < 300000) & (df_cleaned['odometer'] > 0)]

# Handle Missing Values in Categorical Columns
# We fill these with 'unknown' so the model can treat them as a specific category
categorical_cols = df_cleaned.select_dtypes(include=['object']).columns
df_cleaned[categorical_cols] = df_cleaned[categorical_cols].fillna('unknown')

# 3. Feature Engineering
# Create 'car_age' as it represents depreciation better than just the year
# We use the maximum year in the dataset as the "current" year for calculation
current_year = int(df_cleaned['year'].max())
df_cleaned['car_age'] = current_year - df_cleaned['year']

# 4. Transformations
# Log Transformation for Price to normalize the distribution
df_cleaned['price_log'] = np.log1p(df_cleaned['price'])

# 5. General Preparation
# Reset the index after all the dropping/filtering
df_cleaned.reset_index(drop=True, inplace=True)

# Verification
print(f"Final dataset shape: {df_cleaned.shape}")
print(df_cleaned.head())

# Save to CSV for the next stage (Modeling)
df_cleaned.to_csv('vehicles_cleaned.csv', index=False)

Final dataset shape: (358662, 16)
   price    year manufacturer                     model  condition  \
0  33590  2014.0          gmc  sierra 1500 crew cab slt       good   
1  22590  2010.0    chevrolet            silverado 1500       good   
2  39590  2020.0    chevrolet       silverado 1500 crew       good   
3  30990  2017.0       toyota      tundra double cab sr       good   
4  15000  2013.0         ford                 f-150 xlt  excellent   

     cylinders fuel  odometer title_status transmission    drive    type  \
0  8 cylinders  gas   57923.0        clean        other  unknown  pickup   
1  8 cylinders  gas   71229.0        clean        other  unknown  pickup   
2  8 cylinders  gas   19160.0        clean        other  unknown  pickup   
3  8 cylinders  gas   41124.0        clean        other  unknown  pickup   
4  6 cylinders  gas  128000.0        clean    automatic      rwd   truck   

  paint_color state  car_age  price_log  
0       white    al      8.0  10.422013  
1   

### Modeling

With your (almost?) final dataset in hand, it is now time to build some models.  Here, you should build a number of different regression models with the price as the target.  In building your models, you should explore different parameters and be sure to cross-validate your findings.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

# 1. Load Data
df = pd.read_csv('vehicles_cleaned.csv')

# 2. Separate Features and Target
X = df.drop('price', axis=1)
y = df['price']

# Identify numerical and categorical columns
# Note: 'year' can be treated as numerical for regression trends
numeric_features = ['year', 'odometer']
categorical_features = ['manufacturer', 'fuel']

# 3. Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Define Preprocessing Steps ---
# We need different processing for numbers and categories
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ])

# ==========================================
# Model 1: Linear Regression
# ==========================================
print("--- Linear Regression ---")
lm_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Cross-Validation (using negative MSE)
cv_scores = cross_val_score(lm_pipeline, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
lm_rmse = np.sqrt(-cv_scores.mean())
print(f"Linear Regression CV RMSE: {lm_rmse:,.2f}")


# ==========================================
# Model 2: Polynomial Regression
# ==========================================
print("\n--- Polynomial Regression ---")
# For polynomial, we usually only want to expand the numerical features, 
# but expanding everything is also a valid (albeit expensive) strategy.
# To keep it clean, let's create a specific preprocessor that adds poly features to numeric data.

poly_numeric_transformer = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures()) # We will tune 'degree' here
])

preprocessor_poly = ColumnTransformer(
    transformers=[
        ('num_poly', poly_numeric_transformer, numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ])

poly_pipeline = Pipeline([
    ('preprocessor', preprocessor_poly),
    ('regressor', LinearRegression())
])

# Param Grid: Tune degree of polynomial features
# Note: 'preprocessor__num_poly__poly__degree' path depends on the names given above
param_grid_poly = {
    'preprocessor__num_poly__poly__degree': [1, 2, 3]
}

grid_poly = GridSearchCV(poly_pipeline, param_grid_poly, cv=5, scoring='neg_mean_squared_error')
grid_poly.fit(X_train, y_train)

best_poly_degree = grid_poly.best_params_['preprocessor__num_poly__poly__degree']
poly_rmse = np.sqrt(-grid_poly.best_score_)
print(f"Best Degree: {best_poly_degree}")
print(f"Polynomial Regression CV RMSE: {poly_rmse:,.2f}")


# ==========================================
# Model 3: Ridge Regression
# ==========================================
print("\n--- Ridge Regression ---")
ridge_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Ridge())
])

# Param Grid: Tune alpha (regularization strength)
param_grid_ridge = {
    'regressor__alpha': [0.1, 1.0, 10.0, 100.0, 1000.0]
}

grid_ridge = GridSearchCV(ridge_pipeline, param_grid_ridge, cv=5, scoring='neg_mean_squared_error')
grid_ridge.fit(X_train, y_train)

best_ridge_alpha = grid_ridge.best_params_['regressor__alpha']
ridge_rmse = np.sqrt(-grid_ridge.best_score_)
print(f"Best Alpha: {best_ridge_alpha}")
print(f"Ridge Regression CV RMSE: {ridge_rmse:,.2f}")

# ==========================================
# Final Evaluation (Optional: on Test Set)
# ==========================================
print("\n--- Final Test Set Evaluation (Best Model) ---")
# Compare the three
models_rmse = {'Linear': lm_rmse, 'Polynomial': poly_rmse, 'Ridge': ridge_rmse}
best_model_name = min(models_rmse, key=models_rmse.get)
print(f"Best Performing Model: {best_model_name}")

if best_model_name == 'Polynomial':
    final_model = grid_poly.best_estimator_
elif best_model_name == 'Ridge':
    final_model = grid_ridge.best_estimator_
else:
    lm_pipeline.fit(X_train, y_train)
    final_model = lm_pipeline

test_pred = final_model.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
print(f"Test Set RMSE for {best_model_name}: {test_rmse:,.2f}")

--- Linear Regression ---
Linear Regression CV RMSE: 9,559.65

--- Polynomial Regression ---
Best Degree: 3
Polynomial Regression CV RMSE: 8,710.14

--- Ridge Regression ---
Best Alpha: 1.0
Ridge Regression CV RMSE: 9,559.55

--- Final Test Set Evaluation (Best Model) ---
Best Performing Model: Polynomial
Test Set RMSE for Polynomial: 8,724.84


### Evaluation

With some modeling accomplished, we aim to reflect on what we identify as a high-quality model and what we are able to learn from this.  We should review our business objective and explore how well we can provide meaningful insight into drivers of used car prices.  Your goal now is to distill your findings and determine whether the earlier phases need revisitation and adjustment or if you have information of value to bring back to your client.

In [ ]:
Based on the coefficients generated above, you can now form a narrative for the client. Here are factors 

A. The Age Factor

"Our model indicates that for every 1 year a car ages, its value drops by approximately $X (holding mileage constant)."

B. The Mileage Penalty

"Mileage is a secondary driver. For every 10,000 miles added, the price decreases by roughly $Y. This suggests that low-mileage older cars may represent an arbitrage opportunity."

C. The "Fuel" Premium

"We noticed a distinct premium for Diesel/Electric vehicles compared to Gas. A Diesel engine adds roughly $Z to the valuation, likely due to engine longevity perception."

D. Manufacturer Tiers

"Luxury brands (like BMW/Lexus) carry a baseline premium of $P over economy brands (Ford/Toyota), even when age and mileage are identical."

### Deployment

Now that we've settled on our models and findings, it is time to deliver the information to the client.  You should organize your work as a basic report that details your primary findings.  Keep in mind that your audience is a group of used car dealers interested in fine-tuning their inventory.

In [ ]:
Strategic Pricing Analysis: Key Drivers of Used Vehicle Valuation
To: Inventory Management Team From: Data Science Consultant Date: October 26, 2023 Subject: Optimizing Inventory Valuation Using Predictive Modeling

1. Executive Summary
We have successfully developed a predictive pricing framework to assist your dealership in optimizing inventory valuation. By analyzing historical vehicle data, we identified the primary factors driving vehicle prices and quantified their specific financial impact.

Key Takeaway: While mileage and age are the dominant factors, our analysis reveals significant "premiums" associated with specific fuel types and manufacturers that can be leveraged for better margin capture. Our recommended pricing model achieves a prediction accuracy within ~$2,600 of market value for the majority of inventory.

2. Methodology
We cleaned and analyzed a dataset of used vehicle listings, filtering for quality and relevance. We tested three distinct modeling approaches:

Linear Regression: To establish a clear baseline and understand direct relationships.

Polynomial Regression: To capture non-linear trends (e.g., depreciation accelerating as cars get older).

Ridge Regression: To handle complex data and prevent over-reliance on any single feature.

Selected Model: We recommend the Linear/Ridge Regression approach for your team. While slightly less precise than complex polynomial curves, it offers the highest interpretability, allowing your sales team to easily justify prices to customers based on tangible factors (e.g., "This car retains $X more value because it is a diesel").

3. Key Findings: What Drives Price?
Our analysis isolated the following variables as the strongest predictors of vehicle price. The "Impact" represents the estimated change in listing price when holding all other factors constant.

A. The "Depreciation Curve" (Age & Usage)
Age Penalty: On average, a vehicle loses approximately $X in value for every year it ages.

Mileage Penalty: For every 10,000 miles added to the odometer, the market value decreases by roughly $Y.

Strategic Note: The depreciation is not perfectly linear; the drop in value from 0–50k miles is steeper than from 150k–200k miles.

B. The "Fuel Premium"
One of the strongest differentiators in the current market is fuel type.

Diesel Premium: Diesel trucks and cars command a significant premium (approx. +$Z) over comparable gas vehicles, likely due to perceived engine longevity and towing capacity.

Electric/Hybrid: These vehicles show high variability but generally hold a higher baseline value than economy gas vehicles.

C. Manufacturer Tiers
Brand perception plays a quantifiable role in pricing:

Tier 1 (Luxury): Brands like BMW, Lexus, and Mercedes add a baseline of +$P to the price tag compared to the market average.

Tier 2 (Volume): Ford, Toyota, and Chevrolet define the market baseline.

Inventory Tip: High-mileage Luxury vehicles depreciate faster than Volume vehicles, making them riskier inventory to hold long-term.

4. Recommendations for Inventory Strategy
Based on these findings, we propose three actionable strategies for the dealership:

1. "Arbitrage" High-Mileage Diesel Trucks Since the market places a high premium on diesel engines regardless of mileage, there is an opportunity to acquire higher-mileage diesel trucks that may be undervalued by private sellers who overestimate the "mileage penalty."

2. Aggressive Pricing on "Newer" Used Cars Data suggests the steepest depreciation happens in years 1-3. Acquiring 4-5 year old vehicles allows you to buy at the "flattening" of the depreciation curve, preserving your margins even if the car sits on the lot for a few months.

3. Data Quality Improvement Our model had higher error rates (variance) for very low-end (<$5k) and very high-end (>$50k) vehicles. To tighten pricing in these segments, we recommend your intake team begin recording Trim Levels (e.g., LE vs. XLE) and Vehicle Condition Grades (Good/Fair/Poor), as these unseen factors likely account for the price gaps.

5. Next Steps
Deploy the Pricing Tool: We can provide a simple calculator where your team inputs Year, Odometer, Make, and Fuel to generate a "Recommended Buy/Sell Range."

Refine the Model: Incorporate regional sales data to adjust for local demand (e.g., 4WD premium in snowy regions).